# Clase 4 — De comprensión a preparación: limpia y entiende tus datos

**Duración estimada: 120 minutos**

## Objetivos de aprendizaje

- Aplicar estadística descriptiva (univariantes, correlaciones, histogramas) para entender un dataset.
- Detectar y decidir qué hacer con valores ausentes o mal codificados.
- Limpiar y transformar un dataset real con pandas (nombres de columnas, valores inconsistentes, categorías poco frecuentes).
- Explicar qué es feature engineering y por qué importa.

En la Clase 3 cargaste el dataset de 57.691 recetas de cocina en el DataFrame `recipes`. En esta clase vas a hacer dos cosas con él: primero, **entenderlo** (Data Understanding: ¿estos datos representan bien el problema de identificar la cocina de una receta a partir de sus ingredientes?), y segundo, **limpiarlo y transformarlo** (Data Preparation) para dejarlo listo para modelar en la próxima clase.

En paralelo, vas a ver cómo estas mismas dos etapas se aplicaron al caso real de la aseguradora de salud y los pacientes con insuficiencia cardíaca congestiva (CHF) — ese caso no tiene código porque los datos hospitalarios no son públicos, pero la lógica es idéntica a la que vas a aplicar aquí con código real.

## 0. Recordatorio: cargar el dataset

Esto ya lo hiciste en la Clase 3. Lo repetimos aquí para que este notebook sea autocontenido y puedas ejecutarlo de principio a fin sin depender de otro notebook.

In [ ]:
import pandas as pd  # librería estándar para trabajar con datos en tablas (DataFrames)
pd.set_option('display.max_columns', None)  # muestra todas las columnas al imprimir
import numpy as np  # librería para operaciones numéricas y arrays
import re  # librería para expresiones regulares (búsqueda de patrones en texto)

recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

print("Data read into dataframe!")

In [ ]:
recipes.head()  # primeras 5 filas, para recordar la forma del dataset

In [ ]:
recipes.shape  # (filas, columnas) -> (57691, 385): 57.691 recetas, 384 ingredientes + 1 columna de cocina

## 1. Data Understanding: estadística descriptiva

La pregunta de esta etapa es: **¿estos datos representan bien el problema que quieres resolver?**

Las tres herramientas clásicas son:

1. **Univariantes**: media, mediana, mínimo, máximo, desviación estándar de cada variable.
2. **Correlaciones por pares**: para detectar variables redundantes.
3. **Histogramas**: para ver distribuciones y decidir cómo consolidar categorías.

En un dataset como `recipes`, donde casi todas las columnas de ingredientes son binarias (0/1) una vez limpiadas, la estadística univariante más útil es simplemente "¿en qué porcentaje de recetas aparece este ingrediente?" — a eso llegaremos más adelante en este mismo notebook (la sección de perfiles por cocina). Antes de eso, empecemos por lo más básico: comprobar que los ingredientes que esperamos encontrar existen de verdad como columnas.

### 1.1 ¿Existen los ingredientes que esperamos?

Usamos expresiones regulares (`re`) para buscar, dentro de los nombres de columna, coincidencias parciales con "rice" (arroz), "wasabi" y "soy" (soja).

In [ ]:
ingredients = list(recipes.columns.values)  # lista con todos los nombres de columna

print([match.group(0) for ingredient in ingredients
       for match in [(re.compile(".*(rice).*")).search(ingredient)] if match])

print([match.group(0) for ingredient in ingredients
       for match in [(re.compile(".*(wasabi).*")).search(ingredient)] if match])

print([match.group(0) for ingredient in ingredients
       for match in [(re.compile(".*(soy).*")).search(ingredient)] if match])

Confirmamos que existen las columnas `rice`, `wasabi` y `soy_sauce`.

**Pregunta para guardar:** si una receta contiene arroz + wasabi + salsa de soja a la vez, ¿podríamos afirmar con confianza que es cocina japonesa? Vamos a poner esa pregunta a prueba más adelante, con datos reales, después de limpiar el dataset.

### 1.2 El matiz de los valores ausentes

Antes de seguir, un inciso conceptual importante (sin código, porque `recipes` no tiene este problema exacto, pero sí lo vas a encontrar en datasets reales): cuando una variable tiene valores ausentes, "ausente" puede significar cosas muy distintas —"no", "cero", "no lo sabemos"— o puede esconder un valor inválido disfrazado de válido, como un "999" en una columna de edad que en realidad significa "desconocido". Si no lo detectas, tu programa lo trata como un dato numérico legítimo y distorsiona cualquier estadística que calcules. Por eso el primer paso al recibir un dataset nuevo siempre es mirar sus estadísticas univariantes (mínimo, máximo, media) antes de confiar en ellas ciegamente.

### 1.3 El caso CHF: cuando comprender los datos te obliga a volver atrás

*(Caso narrativo, sin código — estos datos hospitalarios no están disponibles públicamente.)*

En el caso de la aseguradora de salud, el equipo definió inicialmente una "admisión por CHF" como cualquier admisión con un **diagnóstico principal** de insuficiencia cardíaca congestiva. Al aplicar estadística descriptiva sobre los datos recolectados, descubrieron que esa definición **no capturaba todas las admisiones por CHF que la experiencia clínica esperaba**. La solución no fue forzar el modelo con datos incompletos: el equipo **volvió a la etapa de Data Collection** y amplió la definición para incluir también diagnósticos secundarios y terciarios de CHF. Este es el ejemplo perfecto de que la metodología es un ciclo, no una línea recta: cuanto más trabajas con el problema, más aprendes, y ese aprendizaje refina etapas anteriores.

## 2. Data Preparation: limpiar `recipes`

Data Preparation es como lavar verduras recién compradas: quitar la tierra y las imperfecciones antes de cocinar. También es como picar una cebolla fina para que su sabor se reparta mejor por la salsa — transformar los datos para que sea más fácil trabajar con ellos.

Un dato importante: junto con la recolección y la comprensión, la preparación de datos suele consumir entre el **70% y el 90%** del tiempo total de un proyecto de ciencia de datos. Automatizar parte de este proceso en la base de datos puede bajarlo hasta el 50%.

Empecemos por mirar la variable de cocina con `.value_counts()`, un método de pandas que cuenta cuántas veces aparece cada valor distinto de una columna.

In [ ]:
recipes["country"].value_counts()  # tabla de frecuencias de la columna "country" 

Este vistazo destapa cuatro problemas de calidad de datos:

1. La columna se llama `country`, pero en realidad contiene el **tipo de cocina** — el nombre está mal puesto.
2. Los nombres de cocina mezclan mayúsculas y minúsculas de forma inconsistente.
3. Hay cocinas duplicadas como variaciones del nombre del país (ej. "Vietnam" vs "Vietnamese").
4. Algunas cocinas tienen muy pocas recetas.

Vamos a arreglar los cuatro, uno por uno.

### 2.1 Arreglar el nombre de la columna

In [ ]:
column_names = recipes.columns.values  # array con los nombres de columna
column_names[0] = "cuisine"              # la primera columna pasa de "country" a "cuisine"
recipes.columns = column_names           # reasigna los nombres corregidos

recipes

### 2.2 Minúsculas, para que "Italian" e "italian" no cuenten como valores distintos

In [ ]:
recipes["cuisine"] = recipes["cuisine"].str.lower()

### 2.3 Unificar nombres duplicados

`.loc[condición, columna] = valor` es el patrón de pandas para: localizar las filas que cumplen una condición, y sobrescribir el valor de una columna en esas filas. Aquí lo usamos fila por país para dejar un solo nombre de cocina por cada una.

In [ ]:
recipes.loc[recipes["cuisine"] == "austria", "cuisine"] = "austrian"
recipes.loc[recipes["cuisine"] == "australia", "cuisine"] = "australian"
recipes.loc[recipes["cuisine"] == "belgium", "cuisine"] = "belgian"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "canada", "cuisine"] = "canadian"
recipes.loc[recipes["cuisine"] == "netherlands", "cuisine"] = "dutch"
recipes.loc[recipes["cuisine"] == "france", "cuisine"] = "french"
recipes.loc[recipes["cuisine"] == "germany", "cuisine"] = "german"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"
recipes.loc[recipes["cuisine"] == "indonesia", "cuisine"] = "indonesian"
recipes.loc[recipes["cuisine"] == "iran", "cuisine"] = "iranian"
recipes.loc[recipes["cuisine"] == "italy", "cuisine"] = "italian"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "israel", "cuisine"] = "jewish"
recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "lebanon", "cuisine"] = "lebanese"
recipes.loc[recipes["cuisine"] == "malaysia", "cuisine"] = "malaysian"
recipes.loc[recipes["cuisine"] == "mexico", "cuisine"] = "mexican"
recipes.loc[recipes["cuisine"] == "pakistan", "cuisine"] = "pakistani"
recipes.loc[recipes["cuisine"] == "philippines", "cuisine"] = "philippine"
recipes.loc[recipes["cuisine"] == "scandinavia", "cuisine"] = "scandinavian"
recipes.loc[recipes["cuisine"] == "spain", "cuisine"] = "spanish_portuguese"
recipes.loc[recipes["cuisine"] == "portugal", "cuisine"] = "spanish_portuguese"
recipes.loc[recipes["cuisine"] == "switzerland", "cuisine"] = "swiss"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "turkey", "cuisine"] = "turkish"
recipes.loc[recipes["cuisine"] == "vietnam", "cuisine"] = "vietnamese"
recipes.loc[recipes["cuisine"] == "uk-and-ireland", "cuisine"] = "uk-and-irish"
recipes.loc[recipes["cuisine"] == "irish", "cuisine"] = "uk-and-irish"

recipes

### 2.4 Eliminar cocinas con menos de 50 recetas

Primero recalculamos la tabla de frecuencias ya con los nombres unificados, y nos quedamos con las cocinas que superan el umbral.

In [ ]:
recipes_counts = recipes["cuisine"].value_counts()
recipes_counts

In [ ]:
cuisines_indices = recipes_counts > 50  # True/False según si la cocina supera 50 recetas
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])

print("Cocinas que se conservan:")
print(cuisines_to_keep)

`.isin()` es un método de pandas que comprueba, fila por fila, si el valor de una columna está dentro de una lista dada. Lo usamos para filtrar el DataFrame y quedarnos solo con las cocinas que superan el umbral.

In [ ]:
rows_before = recipes.shape[0]
print("Number of rows of original dataframe is {}.".format(rows_before))

recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

rows_after = recipes.shape[0]
print("Number of rows of processed dataframe is {}.".format(rows_after))
print("{} rows removed!".format(rows_before - rows_after))

### 2.5 Convertir "Yes"/"No" a 1/0

Casi todas las columnas de ingredientes están codificadas como texto "Yes"/"No". Para poder sumarlas, promediarlas y usarlas en un modelo, las convertimos a números con `.replace()`, que sustituye un valor por otro en todo el DataFrame.

In [ ]:
recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

recipes.head()

## 3. Poniendo a prueba la pregunta: ¿rice + soy + wasabi + seaweed = japonesa?

Con los datos ya limpios y numéricos, filtramos las recetas que combinan cuatro ingredientes típicamente asociados a la cocina japonesa.

In [ ]:
check_recipes = recipes.loc[
    (recipes["rice"] == 1) &
    (recipes["soy_sauce"] == 1) &
    (recipes["wasabi"] == 1) &
    (recipes["seaweed"] == 1)
]
check_recipes

**Respuesta: no.** Junto a recetas japonesas aparecen recetas etiquetadas como "asian" y "east_asian" con exactamente la misma combinación de ingredientes. Un conjunto de ingredientes compartido no basta para identificar una cocina con certeza — necesitas más información, o un modelo capaz de sopesar muchos ingredientes a la vez.

## 4. Ingrediente más popular: cuidado con el sesgo

Sumamos cada columna de ingredientes (ahora que son 1s y 0s, sumar una columna cuenta cuántas veces aparece ese ingrediente en el dataset).

In [ ]:
ing = recipes.iloc[:, 1:].sum(axis=0)  # suma cada columna, saltando la columna 0 ("cuisine")
ing

In [ ]:
ingredient = pd.Series(ing.index.values, index=np.arange(len(ing)))
count = pd.Series(list(ing), index=np.arange(len(ing)))
ing_df = pd.DataFrame(dict(ingredient=ingredient, count=count))
ing_df = ing_df[["ingredient", "count"]]
print(ing_df.to_string())

In [ ]:
ing_df.sort_values(["count"], ascending=False, inplace=True)
ing_df.reset_index(inplace=True, drop=True)
print(ing_df)

Los tres ingredientes más frecuentes son **egg** (20.997), **wheat** (20.757) y **butter** (20.699). Pero antes de sacar conclusiones: el dataset contiene cerca de 40.000 recetas americanas frente a un número mucho menor del resto de cocinas. Este ranking está **sesgado hacia ingredientes típicos de la cocina americana**, simplemente porque hay muchísimas más recetas de ese origen. Un hallazgo estadístico nunca está completo sin preguntarte de dónde viene la muestra.

## 5. Perfil de ingredientes por cocina

Para evitar el sesgo del conteo en bruto, calculamos el **promedio por cocina** con `.groupby()` (agrupa las filas según el valor de una columna) seguido de `.mean()` (promedio de cada columna numérica dentro de cada grupo). Como cada ingrediente es 0 o 1, su promedio dentro de una cocina es directamente el porcentaje de recetas de esa cocina que lo incluyen.

In [ ]:
cuisines = recipes.groupby("cuisine").mean()
cuisines.head()

Por ejemplo: el almendra (`almond`) aparece en el 15,65% de las recetas africanas, y la mantequilla (`butter`) aparece en el 38,11% de las recetas canadienses.

Ahora vamos a automatizar esto: para cada cocina, mostrar sus 4 ingredientes más característicos.

In [ ]:
num_ingredients = 4  # cuántos ingredientes top mostrar por cocina

def print_top_ingredients(row):
    print(row.name.upper())
    row_sorted = row.sort_values(ascending=False) * 100
    top_ingredients = list(row_sorted.index.values)[0:num_ingredients]
    row_sorted = list(row_sorted)[0:num_ingredients]
    for ind, ingredient in enumerate(top_ingredients):
        print("%s (%d%%)" % (ingredient, row_sorted[ind]), end=' ')
    print("\n")

# .apply(..., axis=1) ejecuta print_top_ingredients() una vez por cada fila (cada cocina)
create_cuisines_profiles = cuisines.apply(print_top_ingredients, axis=1)

Esto es feature engineering en acción: no inventamos datos de la nada, usamos lo que ya sabíamos del dataset (que cada columna es un ingrediente binario) para construir una vista nueva —el perfil de ingredientes característicos de cada cocina— que no existía como columna en el CSV original, pero que es mucho más útil para entender el problema y, en la próxima clase, para modelarlo.

## 6. El caso CHF: Data Preparation completa

*(Caso narrativo, sin código.)*

En el caso CHF, Data Preparation empezó por definir con precisión qué contaba como "insuficiencia cardíaca congestiva", usando **códigos de grupos relacionados por diagnóstico** y guía clínica de expertos médicos —porque la CHF es solo uno de varios tipos de insuficiencia cardíaca. Después se definió el criterio de **reingreso**: una ventana de **30 días** después del alta de la admisión inicial (llamada admisión índice), basada en experiencia clínica.

Los registros originales estaban en **formato transaccional** —múltiples registros por paciente, con reclamaciones de médicos, laboratorios, hospitales y servicios clínicos; un solo paciente podía tener cientos o miles de estos registros—. El equipo **agregó** todo esto a **nivel de paciente**, generando columnas nuevas (frecuencia y fecha de visitas recientes, diagnósticos, procedimientos, prescripciones) y considerando **comorbilidades** como diabetes o hipertensión. Todo se combinó con datos demográficos (edad, género, tipo de seguro) para producir **una tabla con un registro por paciente**, con una variable objetivo binaria: reingreso por CHF dentro de 30 días, sí o no.

El resultado final: una **cohorte de 2.343 pacientes**, dividida en un conjunto de entrenamiento y uno de prueba, lista para construir y validar el modelo — que es exactamente lo que vas a hacer en la próxima clase, tanto en el caso narrativo como con el dataset de recetas ya limpio que dejaste preparado en este notebook.

## Resumen de la clase

- Aplicaste estadística descriptiva (univariantes, expresiones regulares para verificar columnas) para entender `recipes`.
- Viste por qué los valores ausentes o inválidos (como un "999" disfrazado de edad válida) necesitan interpretación, no solo detección.
- Limpiaste `recipes` de principio a fin: nombre de columna, minúsculas, nombres duplicados de cocina, cocinas con pocas recetas, y "Yes"/"No" a 1/0.
- Descubriste que rice+soy+wasabi+seaweed no identifica de forma única la cocina japonesa, y que el ingrediente "más popular" del dataset está sesgado hacia la cocina americana.
- Construiste un perfil de ingredientes por cocina con `.groupby().mean()` — un ejemplo real de feature engineering.
- Recorriste el caso CHF de Data Understanding (la definición de CHF que hubo que ampliar) y Data Preparation (definición clínica, ventana de 30 días, agregación a nivel de paciente, cohorte final de 2.343 pacientes).

El DataFrame `recipes` y `cuisines` que dejaste en este notebook son exactamente los que vas a usar en el notebook de práctica de esta clase, y en la Clase 5 para construir tu primer árbol de decisión.